Selects 2 Hubs, assigns every Tract to an open Hub within 1.5 km, and minimizes the population weighted distance.

In [ ]:
from pathlib import Path
import pandas as pd
import gurobipy as gp
from gurobipy import GRB

#BASE = Path(__file__).resolve().parent
BASE = Path.cwd()
DATA = BASE / "data"
RESULTS = BASE / "results"
RESULTS.mkdir(exist_ok=True)

P = 2
R = 1500
T = 500

zones = pd.read_csv(
    DATA / "microhub_Tracts_With_2020_Population_Demand.csv",
    dtype={"Zone_ID": str}
)

sites = pd.read_csv(
    DATA / "microhub_COLP_202_Sites_Truck_Route_Proximity.csv"
)

dist = pd.read_csv(
    DATA / "microhub_310x202_Hub_Zone_Distances_m.csv",
    dtype={"Zone_ID": str}
).set_index("Zone_ID")


# Select study area
# --------------------------------------------------
neighborhoods = ["Lower East Side", "East Village"]

zones = zones[
    zones["Neighborhood"].isin(neighborhoods)
    & (zones["Population_2020"] > 0)
].copy()

# Truck route eligibility
sites = sites[sites["NEAREST_TRUCK_ROUTE_M"] <= T].copy()

# Select relevant distances
dist = dist.loc[zones["Zone_ID"], sites["SITE_ID"]]

# Remove sites unable to serve any tract
valid_sites = dist.columns[(dist <= R).any(axis=0)]

dist = dist[valid_sites]

sites = sites[
    sites["SITE_ID"].isin(valid_sites)
].copy()

# Model sets and weights
# --------------------------------------------------
I = dist.index.tolist()
J = dist.columns.tolist()

weights = (
    zones.set_index("Zone_ID")
    .loc[I, "Population_2020"]
)

# Allow only assignments within 1.5 km
A = [
    (i, j)
    for i in I
    for j in J
    if dist.loc[i, j] <= R
]

print("Demand zones:", len(I))
print("Candidate sites:", len(J))
print("Eligible assignments:", len(A))


# Build optimization model
# --------------------------------------------------
model = gp.Model("Uncapacitated_pMedian")

# Decision variables
y = model.addVars(J, vtype=GRB.BINARY, name="open")

x = model.addVars(A, vtype=GRB.BINARY, name="assign")

# Objective function
model.setObjective(
    gp.quicksum(
        weights.loc[i] * dist.loc[i, j] * x[i, j]
        for i, j in A
    ),
    GRB.MINIMIZE
)

# C1: Assign every tract to exactly one hub
for i in I:
    model.addConstr(
        gp.quicksum(
            x[i, j] for j in J if (i, j) in x
        ) == 1,
        name=f"assignment_{i}"
    )

# C2: Assignment only to open hubs
for i, j in A:
    model.addConstr(
        x[i, j] <= y[j]
    )

# C3: Open exactly 2 hubs
model.addConstr(
    gp.quicksum(y[j] for j in J) == P,
    name="hub_count"
)


# Solve
# --------------------------------------------------
model.optimize()

if model.Status != GRB.OPTIMAL:
    raise RuntimeError(
        f"Optimization failed. Status: {model.Status}"
    )


# Extract solution
# --------------------------------------------------
selected_hubs = [j for j in J if y[j].X > 0.5]

assignments = []

for i in I:
    for j in J:
        if (i, j) in x and x[i, j].X > 0.5:
            assignments.append({
                "Zone_ID": i,
                "Site_ID": j,
                "Population": weights.loc[i],
                "Distance_m": dist.loc[i, j]
            })

assignment_df = pd.DataFrame(assignments)

weighted_mean = (model.ObjVal / weights.sum())

print("Selected hubs:", selected_hubs)
print("Objective value:", model.ObjVal)
print("Weighted mean distance:", weighted_mean)


# Sanity checks
# --------------------------------------------------
assert len(selected_hubs) == P
assert len(assignment_df) == len(I)
assert assignment_df["Zone_ID"].nunique() == len(I)
assert set(assignment_df["Site_ID"]).issubset(set(selected_hubs))
assert (assignment_df["Distance_m"] <= R + 1e-6).all()
assert (
    sites.loc[
        sites["SITE_ID"].isin(selected_hubs),
        "NEAREST_TRUCK_ROUTE_M"
    ] <= T
).all()

recomputed_objective = (
    assignment_df["Population"]
    * assignment_df["Distance_m"]
).sum()

assert abs(recomputed_objective - model.ObjVal) < 1e-4
print("All sanity checks passed.")

# Save results
# --------------------------------------------------
assignment_df.to_csv(
    RESULTS / "assignments.csv",
    index=False
)

sites[
    sites["SITE_ID"].isin(selected_hubs)
].to_csv(
    RESULTS / "selected_hubs.csv",
    index=False
)

Demand zones: 24
Candidate sites: 19
Eligible assignments: 204
Set parameter Username
Set parameter LicenseID to value 2871750
Academic license - for non-commercial use only - expires 2027-10-05
Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-1355U, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 229 rows, 223 columns and 631 nonzeros (Min)
Model fingerprint: 0x2ee0d8ea
Model has 204 linear objective coefficients
Variable types: 0 continuous, 223 integer (223 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [2e+05, 1e+07]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+00]

Presolve time: 0.02s
Presolved: 229 rows, 223 columns, 631 nonzeros
Variable types: 0 continuous, 223 integer (223 binary)
Found heuristic solution: objective 8.772484e+07

Root relaxation: objective 8.35